## Import Statements

In [1]:
import xarray as xr
import dask.array as da
from lightgbm import LGBMRegressor
from dask.distributed import LocalCluster, Client, performance_report
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import psutil
import os
from dask.diagnostics import ProgressBar
import warnings
from openeo.local import LocalConnection
from ipyleaflet import Map, DrawControl, Rectangle, LayerGroup
from IPython.display import display
import ipywidgets as widgets
from math import floor, ceil
from ipywidgets import widgets, Layout
from datetime import date, timedelta


warnings.filterwarnings(
    "ignore",
    module="sklearn.utils.validation"
)

In [2]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Access AWS credentials
access_key = os.getenv("ACCESS_KEY")
secret_key = os.getenv("SECRET_KEY")

In [3]:
## Constants
STAC_URLS = {
    # Static ERA5 and EMO1 collections
    "ERA5_T2M_SSRD_TP": "https://stac.intertwin.fedcloud.eu/collections/ERA5_T2M_SSRD_TP",
    "ERA5_PRESSURE": "https://stac.intertwin.fedcloud.eu/collections/ERA5_PRESSURE",
    "EMO1_TA24_PR_RG_PET_DAILY": "https://stac.intertwin.fedcloud.eu/collections/EMO1_TA24_PR_RG_PET_DAILY",
    "EMO1_DEM": "https://stac.intertwin.fedcloud.eu/collections/EMO1_DEM",
    
    # Dynamic SEAS5 collections (use format() or f-strings when needed)
    "SEAS5_SINGLE": "https://stac.intertwin.fedcloud.eu/collections/SINGLE_LEVELS_DAILY_SEAS5_{init}",
    "SEAS5_PRESSURE": "https://stac.intertwin.fedcloud.eu/collections/PRESSURE_LEVELS_DAILY_SEAS5_{init}"
}

In [4]:

# Bounding box limits
min_lat, max_lat = 40, 52
min_lng, max_lng = 2, 20

# Create base map
m = Map(center=((min_lat + max_lat) / 2, (min_lng + max_lng) / 2), zoom=6)

# Add visible constraint boundary (green)
bounds_layer = Rectangle(bounds=((min_lat, min_lng), (max_lat, max_lng)),
                         color="green", fill_opacity=0.05)
m.add_layer(bounds_layer)

# Drawing control (rectangle only)
draw_control = DrawControl(rectangle={"shapeOptions": {"color": "#0000FF"}})
draw_control.circle = {}
draw_control.polyline = {}
draw_control.polygon = {}
draw_control.marker = {}
m.add_control(draw_control)

# Layers to display rectangles
drawn_rect_layer = LayerGroup()
rounded_rect_layer = LayerGroup()
m.add_layer(drawn_rect_layer)
m.add_layer(rounded_rect_layer)

# Output variables
user_bbox = {}
rounded_bbox = {}

# Button for confirmation
confirm_button = widgets.Button(description="✅ Confirm Selection", disabled=True, button_style='success')

def floor_half(val):
    return (val // 0.5) * 0.5

def ceil_half(val):
    return ((val + 0.5 - 1e-9) // 0.5) * 0.5


# Callback when user draws
def handle_draw(target, action, geo_json):
    global user_bbox, rounded_bbox

    coords = geo_json['geometry']['coordinates'][0]
    lats = [coord[1] for coord in coords]
    lngs = [coord[0] for coord in coords]

    user_bbox = {
        "west": min(lngs),
        "east": max(lngs),
        "south": min(lats),
        "north": max(lats)
    }

    if (user_bbox["west"] < min_lng or user_bbox["east"] > max_lng or
        user_bbox["south"] < min_lat or user_bbox["north"] > max_lat):
        print("❌ Drawn bounding box exceeds allowed limits! Try again.")
        confirm_button.disabled = True
        return

    # Accommodative outward rounding
    rounded_bbox  = {
        "west": floor_half(user_bbox["west"]),
        "east": ceil_half(user_bbox["east"]),
        "south": floor_half(user_bbox["south"]),
        "north": ceil_half(user_bbox["north"])
    }

    # Draw both rectangles
    drawn_rect_layer.clear_layers()
    rounded_rect_layer.clear_layers()

    user_rect = Rectangle(
        bounds=((user_bbox["south"], user_bbox["west"]),
                (user_bbox["north"], user_bbox["east"])),
        color="blue",
        fill_opacity=0.05
    )
    rounded_rect = Rectangle(
        bounds=((rounded_bbox ["south"], rounded_bbox ["west"]),
                (rounded_bbox ["north"], rounded_bbox ["east"])),
        color="red",
        fill_opacity=0.1
    )

    drawn_rect_layer.add_layer(user_rect)
    rounded_rect_layer.add_layer(rounded_rect)

    confirm_button.disabled = False

draw_control.on_draw(handle_draw)

# Display map and confirmation button
display(m)
display(confirm_button)

# Optional: define callback if you want further logic
def on_confirm_clicked(b):
    print("✅ Selection confirmed.")
    print("Use `user_bbox` and `rounded_bbox` as needed.")

confirm_button.on_click(on_confirm_clicked)


Map(center=[46.0, 11.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out…

Button(button_style='success', description='✅ Confirm Selection', disabled=True, style=ButtonStyle())

In [5]:
# Print both
print("🟦 User-drawn bounding box:")
print(user_bbox)
print("🟥 Rounded bounding box:")
print(rounded_bbox)

#spatial_extent = rounded_bbox
spatial_extent = {'west': 10.5, 'east': 11.5, 'south': 45.5, 'north': 46.5}


🟦 User-drawn bounding box:
{}
🟥 Rounded bounding box:
{}


In [6]:
# Initialize temporal_extent
temporal_extent = ['2020-01-01', '2021-01-01']

# Create date pickers with reasonable defaults
start_date_picker = widgets.DatePicker(
    description='Start Date:',
    value=date(2000, 1, 1),
    layout=Layout(width='300px')
)

end_date_picker = widgets.DatePicker(
    description='End Date:',
    value=date(2020, 12, 31),
    layout=Layout(width='300px')
)

# Create output widget to display the selected range
output = widgets.Output()

# Button to confirm selection
button = widgets.Button(description="Set Temporal Extent")

def on_button_click(b):
    with output:
        temporal_extent.clear()
        start_date = start_date_picker.value
        end_date = end_date_picker.value + timedelta(days=1)  # Add +1 day to make end date exclusive
        
        # Validate date range
        if start_date > end_date_picker.value:  # Compare with original end date before adjustment
            print("❌ Error: Start date must be before end date!")
            return
            
        temporal_extent.extend([
            start_date.strftime("%Y-%m-%d"),
            end_date.strftime("%Y-%m-%d")
        ])
        print(f"Training Period extent set to: {temporal_extent}")
        print(f"Note: End date has been adjusted +1 day to {end_date.strftime('%Y-%m-%d')} for inclusive coverage")

button.on_click(on_button_click)

# Display the widgets
display(widgets.VBox([
    widgets.HTML("<h3>Select Training Period</h3>"),
    start_date_picker,
    end_date_picker,
    button,
    output
]))

In [7]:
# Initialize target_variable
target_variable = None

# Create radio buttons
radio = widgets.RadioButtons(
    options=["t2m", "ssrd", "tp"],
    description='Target Variable:',
    disabled=False,
    layout=Layout(width='200px')
)

# Create output widget
output = widgets.Output()

def on_change(change):
    global target_variable
    if change['type'] == 'change' and change['name'] == 'value':
        target_variable = change['new']
        with output:
            output.clear_output()
            print(f"Selected target variable: {target_variable}")

radio.observe(on_change)

# Display the widgets
display(widgets.VBox([
    widgets.HTML("<h3>Select Target Variable</h3>"),
    radio,
    output
]))

In [8]:
if __name__ == '__main__':
    # Initialize Dask cluster and client inside the main block
    cluster = LocalCluster(
        n_workers=14,
        threads_per_worker=1,
        worker_dashboard_address=False,
        diagnostics_port=None
    )
    client = Client(cluster)
    
    # Initialize the local connection
    local_conn = LocalConnection("./")

    # Load the data cube with specified parameters
    era5_single = local_conn.load_stac(
        url=STAC_URLS["ERA5_T2M_SSRD_TP"],
        spatial_extent=spatial_extent,
        temporal_extent=temporal_extent,
    )
    
    era5_pressure = local_conn.load_stac(
        url=STAC_URLS["ERA5_PRESSURE"],
        spatial_extent=spatial_extent,
        temporal_extent=temporal_extent,
        bands=["t_850"]
    )
    
    emo1 = local_conn.load_stac(
        url=STAC_URLS["EMO1_TA24_PR_RG_PET_DAILY"],
        bands=["ta24"],
        spatial_extent=spatial_extent,
        temporal_extent=temporal_extent,
    )

    dem = local_conn.load_stac(
        url=STAC_URLS["EMO1_DEM"],
        spatial_extent=spatial_extent,
        bands=["dem"]
    )

    era5_cube = era5_single.merge_cubes(era5_pressure)
    #resample = dem.resample_spatial(resolution=0.666, method="bilinear", projection="EPSG:4326")
    remap = era5_cube.resample_cube_spatial(dem, method="bilinear")
    dem_expanded = dem.resample_cube_temporal(remap)
    cube = remap.merge_cubes(dem_expanded)
    emo1 = emo1.rename_labels(dimension="bands", target=["target_dataset"], source=["ta24"])
    recube = cube.merge_cubes(emo1)
    result = recube.process("sin_cos_doy", data=recube)
    merged_era5_recube = recube.merge_cubes(result)
    print("REACHED HERE!")
    
    
    # to go to raster_to_stac as UUID_X.zarr in the local

REACHED HERE!


In [9]:
merged_era5_recube

In [10]:
#merged_era5_recube.execute()

## SEAS5 Initialisation 

In [11]:
init = "AUGUST_2021"

In [12]:
if __name__ == '__main__':
    # Initialize Dask cluster and client inside the main block
    cluster = LocalCluster(
        n_workers=14,
        threads_per_worker=1,
        worker_dashboard_address=False,
        diagnostics_port=None
    )
    client = Client(cluster)
    
    # Initialize the local connection
    local_conn = LocalConnection("./")

    # Load the data cube with specified parameters
    seas5_single = local_conn.load_stac(
        url=STAC_URLS["SEAS5_SINGLE"].format(init=init),
        spatial_extent=spatial_extent,
        temporal_extent=["2021-08-01", "2021-08-02"],
        bands=["ssrd", "t2m", "tp"]
    )
    
    seas5_pressure = local_conn.load_stac(
        url=STAC_URLS["SEAS5_PRESSURE"].format(init=init),
        spatial_extent=spatial_extent,
        temporal_extent=["2021-08-01", "2021-08-02"],
        bands=["t_850"],
    )

    dem = local_conn.load_stac(
        url=STAC_URLS["EMO1_DEM"],
        spatial_extent=spatial_extent,
        bands=["dem"]
    )

    seas5_cube = seas5_single.merge_cubes(seas5_pressure)
    #resample = dem.resample_spatial(resolution=0.666, method="bilinear", projection="EPSG:4326")
    seas5_remap = seas5_cube.resample_cube_spatial(dem, method="bilinear")
    dem_expanded = dem.resample_cube_temporal(seas5_remap)
    dem_expanded = dem_expanded.rename_dimension(target="y", source="lat")
    dem_expanded = dem_expanded.rename_dimension(target="x", source="lon")
    seas5cube = seas5_remap.merge_cubes(dem_expanded)
    seas = seas5cube.process("sin_cos_doy", data=seas5cube)
    merged_seas5_cube = seas5cube.merge_cubes(seas)
    merged_seas5_cube
    print("REACHED HERE!")
    
    # to go to raster_to_stac as UUID_X.zarr in the local

/home/sdhinakaran/micromamba/envs/zarr_raster2stac/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 44473 instead
  warnings.warn(


REACHED HERE!


In [13]:
merged_seas5_cube

In [14]:
#test_cube = merged_seas5_cube.execute()

In [15]:
#test_cube

## Seeing if i can add raster2stac as a process

In [16]:
UUID = "test0123"

In [17]:
if __name__ == '__main__':
    # Initialize Dask cluster and client inside the main block
    cluster = LocalCluster(
        n_workers=14,
        threads_per_worker=1,
        worker_dashboard_address=False,
        diagnostics_port=None
    )
    client = Client(cluster)
    
    # Initialize the local connection
    local_conn = LocalConnection("./")

    # Load the data cube with specified parameters
    seas5_single = local_conn.load_stac(
        url=STAC_URLS["SEAS5_SINGLE"].format(init=init),
        spatial_extent=spatial_extent,
        temporal_extent=["2021-08-01", "2021-08-02"],
        bands=["ssrd", "t2m", "tp"]
    )
    
    seas5_pressure = local_conn.load_stac(
        url=STAC_URLS["SEAS5_PRESSURE"].format(init=init),
        spatial_extent=spatial_extent,
        temporal_extent=["2021-08-01", "2021-08-02"],
        bands=["t_850"],
    )

    dem = local_conn.load_stac(
        url=STAC_URLS["EMO1_DEM"],
        spatial_extent=spatial_extent,
        bands=["dem"]
    )

    seas5_cube = seas5_single.merge_cubes(seas5_pressure)
    #resample = dem.resample_spatial(resolution=0.666, method="bilinear", projection="EPSG:4326")
    seas5_remap = seas5_cube.resample_cube_spatial(dem, method="bilinear")
    dem_expanded = dem.resample_cube_temporal(seas5_remap)
    dem_expanded = dem_expanded.rename_dimension(target="y", source="lat")
    dem_expanded = dem_expanded.rename_dimension(target="x", source="lon")
    seas5cube = seas5_remap.merge_cubes(dem_expanded)
    seas = seas5cube.process("sin_cos_doy", data=seas5cube)
    merged_seas5_cube = seas5cube.merge_cubes(seas)
    seas_r2s = merged_seas5_cube.process(
        "raster2stac",
        data=merged_seas5_cube,
        item_id=f"TEST_CUBE_{UUID}",
        collection_url="https://stac.intertwin.fedcloud.eu/collections/",
        output_folder="TEST_CUBE",
        description="Testing raster2stac from client",
        write_collection_assets=True,
        keywords=["interTwin", "Zarr", "test"],
        s3_upload=True,
        s3_endpoint_url="https://objectstore.eodc.eu:2222",
        bucket_name="rucio",
        bucket_file_prefix="interTwin_EURAC/"
    )
    print("REACHED HERE!")
    
    # to go to raster_to_stac as UUID_X.zarr in the local

/home/sdhinakaran/micromamba/envs/zarr_raster2stac/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 45353 instead
  warnings.warn(


REACHED HERE!


In [18]:
seas_r2s

In [19]:
trial = seas_r2s.execute()
trial

/home/sdhinakaran/micromamba/envs/zarr_raster2stac/lib/python3.11/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 148.33 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


<xarray.DataArray (bands: 7, time: 1, number: 51, y: 60, x: 60)> Size: 5MB
dask.array<getitem, shape=(7, 1, 51, 60, 60), dtype=float32, chunksize=(1, 1, 1, 5, 5), chunktype=numpy.ndarray>
Coordinates:
  * time         (time) datetime64[ns] 8B 2021-08-01
  * number       (number) int32 204B 0 1 2 3 4 5 6 7 ... 43 44 45 46 47 48 49 50
  * y            (y) float64 480B 46.49 46.48 46.46 46.44 ... 45.54 45.53 45.51
  * x            (x) float64 480B 10.51 10.52 10.54 10.56 ... 11.46 11.47 11.49
  * bands        (bands) <U7 196B 'ssrd' 't2m' 'tp' ... 'sin_doy' 'cos_doy'
    spatial_ref  int64 8B 0
Attributes:
    CDI:                        Climate Data Interface version 1.9.8 (https:/...
    CDO:                        Climate Data Operators version 1.9.8 (https:/...
    Conventions:                CF-1.5
    GDAL:                       GDAL 3.0.4, released 2020/01/28
    GDAL_AREA_OR_POINT:         Area
    NCO:                        netCDF Operators version 4.9.2 (Homepage = ht...
    crs:                        EPSG:4326
    history:                    Fri Nov 20 14:22:33 2020: ncks -A -v lat,lon ...
    history_of_appended_files:  Fri Nov 20 14:22:33 2020: Appended file /huge...

In [20]:
print("Reached the END OF NOTEBOOK!")

<xarray.Dataset> Size: 5MB
Dimensions:      (time: 1, number: 51, y: 60, x: 60)
Coordinates:
  * time         (time) datetime64[ns] 8B 2021-08-01
  * number       (number) int32 204B 0 1 2 3 4 5 6 7 ... 43 44 45 46 47 48 49 50
  * y            (y) float64 480B 46.49 46.48 46.46 46.44 ... 45.54 45.53 45.51
  * x            (x) float64 480B 10.51 10.52 10.54 10.56 ... 11.46 11.47 11.49
    spatial_ref  int64 8B 0
Data variables:
    ssrd         (time, number, y, x) float32 734kB dask.array<chunksize=(1, 1, 5, 5), meta=np.ndarray>
    t2m          (time, number, y, x) float32 734kB dask.array<chunksize=(1, 1, 5, 5), meta=np.ndarray>
    tp           (time, number, y, x) float32 734kB dask.array<chunksize=(1, 1, 5, 5), meta=np.ndarray>
    t_850        (time, number, y, x) float32 734kB dask.array<chunksize=(1, 1, 5, 5), meta=np.ndarray>
    dem          (time, number, y, x) float32 734kB dask.array<chunksize=(1, 1, 5, 5), meta=np.ndarray>
    sin_doy      (time, number, y, x) float32 734kB dask.array<chunksize=(1, 1, 5, 5), meta=np.ndarray>
    cos_doy      (time, number, y, x) float32 734kB dask.array<chunksize=(1, 1, 5, 5), meta=np.ndarray>
Attributes:
    CDI:                        Climate Data Interface version 1.9.8 (https:/...
    CDO:                        Climate Data Operators version 1.9.8 (https:/...
    Conventions:                CF-1.5
    GDAL:                       GDAL 3.0.4, released 2020/01/28
    GDAL_AREA_OR_POINT:         Area
    NCO:                        netCDF Operators version 4.9.2 (Homepage = ht...
    crs:                        EPSG:4326
    history:                    Fri Nov 20 14:22:33 2020: ncks -A -v lat,lon ...
    history_of_appended_files:  Fri Nov 20 14:22:33 2020: Appended file /huge...